In [1]:
# FIX: rewrite ConsentTable & ConsentValueCheck2 as MANAGED tables (saveAsTable)
# so they register in the lakehouse metastore and the SQL endpoint syncs them.
#
# PREREQ: attach the Dataverse lakehouse as the DEFAULT lakehouse in this notebook:
#   dataverse_esacontact_cds2_workspace_a94a4bf848e144bba608bb2eb51cbe
# (Explorer panel -> Add data items -> that lakehouse -> pin as default)
#
from pyspark.sql import functions as F

DV = "abfss://6988ba37-18e0-4677-82f3-aab0aa8112cc@onelake.dfs.fabric.microsoft.com/ee1e74a0-c7a7-423e-8d8d-d4df8faa1b17/Tables"

# ---- rebuild ConsentTable ----
contact = spark.read.format("delta").load(f"{DV}/contact")
def cslice(col, label):
    return contact.select(
        F.col("contactid").alias("ContactID"),
        F.lit(label).alias("ConsentType"),
        F.col(col).alias("ConsentGiven"),
        F.when(F.col(col) == True, 0).otherwise(1).alias("ConsentGivenNumeric"))
consent = (cslice("donotbulkemail","No Bulk Email")
    .unionByName(cslice("donotbulkpostalmail","No Bulk Postal Mail"))
    .unionByName(cslice("donotemail","No Email"))
    .unionByName(cslice("donotphone","No Phone"))
    .unionByName(cslice("donotpostalmail","No Postal Mail"))
    .unionByName(cslice("msgdpr_donottrack","GDPR Do Not Track")))

spark.sql("DROP TABLE IF EXISTS ConsentTable")
consent.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("ConsentTable")
print("ConsentTable (managed) written:", consent.count(), "rows")

# ---- rebuild ConsentValueCheck2 ----
cpc = spark.read.format("delta").load(f"{DV}/msdynmkt_contactpointconsent4")
cols = cpc.columns
type_col = "msdynmkt_contactpointtype_display" if "msdynmkt_contactpointtype_display" in cols else "msdynmkt_contactpointtype"
val_col  = "msdynmkt_value_display" if "msdynmkt_value_display" in cols else "msdynmkt_value"
cvc2 = (cpc.groupBy(F.col(type_col).alias("MessageType"), F.col(val_col).alias("Response"))
           .agg(F.count(F.lit(1)).alias("RowCount")))

spark.sql("DROP TABLE IF EXISTS ConsentValueCheck2")
cvc2.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("ConsentValueCheck2")
print("ConsentValueCheck2 (managed) written:", cvc2.count(), "rows")

# verify metadata folder now exists
print("\nDone. Both tables written as MANAGED tables - SQL endpoint should now sync them.")